# Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

import warnings
warnings.filterwarnings('ignore')
sns.set()
plt.style.use('ggplot')
%matplotlib inline

In [ ]:
df=pd.read_csv("liver.csv")

In [ ]:
df.head()

Target Column is Dataset

Supervised or Unsupervised

Classification

Binary

In [ ]:
df.shape

In [ ]:
df.columns

In [ ]:
df.describe()

# EDA

In [ ]:
df.info()

In [ ]:
df.dtypes[df.dtypes=='object']

Distribution of Numberical Features

In [ ]:
df.hist(figsize=(15,15), xrot=-45,bins=10)
plt.show()

In [ ]:
df.describe()

In [ ]:
def convertdataset(x):
    if x==2:
        return 0
    return 1
df['Dataset'] = df['Dataset'].map(convertdataset)

In [ ]:
df.head()

In [ ]:
df.Dataset.value_counts()

In [ ]:
df.describe(include=['object'])

# Bar plots for categorical features

In [ ]:
plt.figure(figsize=(5,5))
sns.countplot(y='Gender', data=df)

In [ ]:
df[df['Gender'] == 'Male'][['Dataset','Gender']].head()

In [ ]:
sns.factorplot(x="Age", y="Gender", hue="Dataset", data=df)

In [ ]:
df['Gender'].value_counts()

In [ ]:
# Categorical Value Handling
def convertgender(x):
    if x== 'Male':
        return 0
    else:
        return 1
df['Gender'] = df['Gender'].map(convertgender)

In [ ]:
df.head()

# Correlation

In [ ]:
df.corr()

In [ ]:
# Positive Correlation-> one feature increases other also increases
# Negative Correlation-> one feature increases other decreases
# closer to 0-> weak relationship

In [ ]:
plt.figure(figsize=(10,10))
sns.heatmap(df.corr())

In [ ]:
mask = np.zeros_like(df.corr())
mask[np.triu_indices_from(mask)] = True
plt.figure(figsize=(10,10))
with sns.axes_style("white"):
    ax = sns.heatmap(df.corr()*100, mask=mask, fmt = ".0f", annot=True, lw=1, cmap=ListedColormap(['green','yellow','red','blue']))

In [ ]:
df = df.drop_duplicates()

In [ ]:
df.shape

In [ ]:
# Removing Outlier

In [ ]:
df.columns

In [ ]:
sns.boxplot(df.Aspartate_Aminotransferase)

In [ ]:
sns.boxplot(df.Total_Bilirubin)

In [ ]:
df.Aspartate_Aminotransferase.sort_values(ascending=False).head()

In [ ]:
df = df[df.Aspartate_Aminotransferase<=3000]

In [ ]:
df.shape

In [ ]:
df.Aspartate_Aminotransferase.sort_values(ascending=False).head()

In [ ]:
df = df[df.Aspartate_Aminotransferase<=2500]

In [ ]:
df.shape

In [ ]:
df.isnull().sum()

In [ ]:
df = df.dropna(how='any')

In [ ]:
df.head()

In [ ]:
df.shape

# Machine Learning Model

# Data Preparation

In [ ]:
y=df.Dataset
X=df.drop('Dataset', axis=1)

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC

In [ ]:
X_train, X_test, y_train , y_test = train_test_split(X,y, test_size=0.2, random_state=0, stratify=y)

# Data Standardization

In [ ]:
train_mean = X_train.mean()
train_std = X_train.std()

In [ ]:
X_train = (X_train - train_mean) / train_std

In [ ]:
X_train.describe()

In [ ]:
X_test = (X_test - train_mean) / train_std

In [ ]:
X_test.describe()

# Logistic Regression

In [ ]:
lr = LogisticRegression()
lr.fit(X_train, y_train)

In [ ]:
y_pred = lr.predict(X_test)

In [ ]:
print(accuracy_score(y_train, lr.predict(X_train)))
lr_acc = accuracy_score(y_test, lr.predict(X_test))
print(lr_acc)
print(confusion_matrix(y_test, lr.predict(X_test)))
print(classification_report(y_test, lr.predict(X_test)))

# KNeighborsClassifier

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
knn=KNeighborsClassifier()
knn.fit(X_train, y_train)

In [ ]:
knn.predict(X_test)

In [ ]:
print(accuracy_score(y_train, lr.predict(X_train)))
knn_acc = accuracy_score(y_test, knn.predict(X_test))
print(knn_acc)
print(confusion_matrix(y_test, knn.predict(X_test)))
print(classification_report(y_test, knn.predict(X_test)))

# SVC

In [ ]:
svc= SVC(probability=True)
parameters = {
    'gamma':[0.0001, 0.001, 0.01, 0.1],
    'C':[0.01, 0.05, 0.5, 0.1, 1, 10, 15, 20, 30]
}
grid_search = GridSearchCV(svc, parameters)
grid_search.fit(X_train, y_train)

In [ ]:
grid_search.best_params_

In [ ]:
grid_search.best_score_

In [ ]:
svc= SVC(C=0.01, gamma=0.0001,probability=True)
svc.fit(X_train, y_train)

In [ ]:
print(accuracy_score(y_train, svc.predict(X_train)))
svc_acc = accuracy_score(y_test, svc.predict(X_test))
print(svc_acc)
print(confusion_matrix(y_test, svc.predict(X_test)))
print(classification_report(y_test, svc.predict(X_test)))

# DecisionTree

In [ ]:
dtc = DecisionTreeClassifier()
dtc.fit(X_train, y_train)

In [ ]:
print(accuracy_score(y_train, dtc.predict(X_train)))
dtc_acc = accuracy_score(y_test, dtc.predict(X_test))
print(dtc_acc)
print(confusion_matrix(y_test, dtc.predict(X_test)))
print(classification_report(y_test, dtc.predict(X_test)))

# Hyper Tunning

In [ ]:
grid_parameter = {
    'criterion':['gini','entropy'],
    'max_depth':[3,5,7,10,12,15],
    'splitter':['best','random'],
    'min_samples_leaf':[1,2,3,5,7],
    'min_samples_split':[1,2,3,5,7],
    'max_features':['auto','sqrt','log2']
}
grid_seach_dt = GridSearchCV(dtc, grid_parameter, cv=24, n_jobs=-1, verbose=1)
grid_seach_dt.fit(X_train, y_train)

In [ ]:
grid_seach_dt.best_params_

In [ ]:
grid_seach_dt.best_score_

In [ ]:
dtc = DecisionTreeClassifier(criterion='entropy', max_depth=5, max_features='sqrt', min_samples_leaf=7, min_samples_split=3, splitter='best')
dtc.fit(X_train, y_train)

In [ ]:
print(accuracy_score(y_train, dtc.predict(X_train)))
dtc_acc = accuracy_score(y_test, dtc.predict(X_test))
print(dtc_acc)
print(confusion_matrix(y_test, dtc.predict(X_test)))
print(classification_report(y_test, dtc.predict(X_test)))

# Random  Forest

In [ ]:
rand_clf = RandomForestClassifier(criterion='entropy', max_depth=15, max_features=0.75, min_samples_leaf=7, min_samples_split=3, n_estimators = 130)

In [ ]:
rand_clf.fit(X_train, y_train)

In [ ]:
print(accuracy_score(y_train, rand_clf.predict(X_train)))
rand_clf_acc = accuracy_score(y_test, rand_clf.predict(X_test))
print(rand_clf_acc)
print(confusion_matrix(y_test, rand_clf.predict(X_test)))
print(classification_report(y_test, rand_clf.predict(X_test)))

# Gradient Boosting Classifier

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier

gbc = GradientBoostingClassifier()

parameters = {
    'loss': ['deviance', 'exponential'],
    'learning_rate': [0.001, 0.1, 1, 10],
    'n_estimators': [100, 150, 180, 200]
}

grid_search_gbc = GridSearchCV(gbc, parameters, cv = 20, n_jobs = -1, verbose = 1)
grid_search_gbc.fit(X_train, y_train)

In [ ]:
grid_search_gbc.best_params_

In [ ]:
grid_search_gbc.best_score_

In [ ]:
gbc = GradientBoostingClassifier(learning_rate=0.001, loss='exponential',n_estimators=100)
gbc.fit(X_train , y_train)

In [ ]:
print(accuracy_score(y_train, gbc.predict(X_train)))
gbc_acc = accuracy_score(y_test, gbc.predict(X_test))
print(gbc_acc)
print(confusion_matrix(y_test, gbc.predict(X_test)))
print(classification_report(y_test, gbc.predict(X_test)))

# XGBoost

In [ ]:
from xgboost import XGBClassifier

In [ ]:
xgb = XGBClassifier(objective='binary:logistic', learning_rate = 0.001, max_depth = 100, n_estimators = 300)
xgb.fit(X_train, y_train)

In [ ]:
print(accuracy_score(y_train, xgb.predict(X_train)))
xgb_acc = accuracy_score(y_test, xgb.predict(X_test))
print(xgb_acc)
print(confusion_matrix(y_test, xgb.predict(X_test)))
print(classification_report(y_test, xgb.predict(X_test)))

# Model Comparison

In [ ]:
models = pd.DataFrame({
    'Model':['Logistic Regreesion','KNN', 'SVC', 'Decision Tree Classifier', 'Random Forest Classifier', 'Gradient Boosting Classifer', 'XgBoost'],
    'Score':[100*round(lr_acc, 4), 100*round(knn_acc, 4), 100*round(svc_acc, 4), 100*round(dtc_acc, 4), 100*round(rand_clf_acc, 4), 100*round(gbc_acc, 4),100*round(xgb_acc, 4)]
})

In [ ]:
models.sort_values(by='Score', ascending=False)

In [ ]:
import pickle
model = lr_acc
pickle.dump(model, open("liver.pkl","wb"))

In [ ]:
# 85%
# ANN

In [ ]:
from sklearn import metrics
plt.figure(figsize=(8,5))
models = [
{
    'label': 'LR',
    'model': lr,
},
{
    'label': 'DT',
    'model': dtc,
},
{
    'label': 'SVM',
    'model': svc,
},
{
    'label': 'KNN',
    'model': knn,
},
{
    'label': 'XGBoost',
    'model': xgb,
},
{
    'label': 'RF',
    'model': rand_clf,
},
{
    'label': 'GBDT',
    'model': gbc,
}
]
for m in models:
    model = m['model'] 
    model.fit(X_train, y_train) 
    y_pred=model.predict(X_test) 
    fpr1, tpr1, thresholds = metrics.roc_curve(y_test, model.predict_proba(X_test)[:,1])
    auc = metrics.roc_auc_score(y_test,model.predict(X_test))
    plt.plot(fpr1, tpr1, label='%s - ROC (area = %0.2f)' % (m['label'], auc))

plt.plot([0, 1], [0, 1],'r--')
plt.xlim([-0.01, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('1 - Specificity (False Positive Rate)', fontsize=12)
plt.ylabel('Sensitivity (True Positive Rate)', fontsize=12)
plt.title('ROC - Liver Disease Prediction', fontsize=12)
plt.legend(loc="lower right", fontsize=12)
plt.savefig("roc_liver.jpeg", format='jpeg', dpi=400, bbox_inches='tight')
plt.show()

In [ ]:
from sklearn import metrics
import numpy as np
import matplotlib.pyplot as plt
models = [
{
    'label': 'LR',
    'model': lr,
},
{
    'label': 'DT',
    'model': dtc,
},
{
    'label': 'SVM',
    'model': svc,
},
{
    'label': 'KNN',
    'model': knn,
},
{
    'label': 'XGBoost',
    'model': xgb,
},
{
    'label': 'RF',
    'model': rand_clf,
},
{
    'label': 'GBDT',
    'model': gbc,
}
]

means_roc = []
means_accuracy = [100*round(lr_acc,4), 100*round(dtc_acc,4), 100*round(svc_acc,4), 100*round(knn_acc,4), 100*round(xgb_acc,4), 
                  100*round(rand_clf_acc,4), 100*round(gbc_acc,4)]

for m in models:
    model = m['model'] 
    model.fit(X_train, y_train) 
    y_pred=model.predict(X_test) 
    fpr1, tpr1, thresholds = metrics.roc_curve(y_test, model.predict_proba(X_test)[:,1])
    auc = metrics.roc_auc_score(y_test,model.predict(X_test))
    auc = 100*round(auc,4)
    means_roc.append(auc)

print(means_accuracy)
print(means_roc)

# data to plot
n_groups = 7
means_accuracy = tuple(means_accuracy)
means_roc = tuple(means_roc)

# create plot
fig, ax = plt.subplots(figsize=(8,5))
index = np.arange(n_groups)
bar_width = 0.35
opacity = 0.8

rects1 = plt.bar(index, means_accuracy, bar_width,
alpha=opacity,
color='mediumpurple',
label='Accuracy (%)')

rects2 = plt.bar(index + bar_width, means_roc, bar_width,
alpha=opacity,
color='rebeccapurple',
label='ROC (%)')

plt.xlim([-1, 8])
plt.ylim([40, 76])

plt.title('Performance Evaluation - Liver Disease Prediction', fontsize=12)
plt.xticks(index, ('   LR', '   DT', '   SVM', '   KNN', 'XGBoost' , '   RF', '   GBDT'), rotation=40, ha='center', fontsize=12)
plt.legend(loc="upper right", fontsize=10)
plt.savefig("PE_liver.jpeg", format='jpeg', dpi=400, bbox_inches='tight')
plt.show()

## 📥 Download Trained Model & Artifacts (Kaggle)

This cell saves the trained classification models (`liver_model.pkl`, `liver_rf_model.pkl`, `liver_lr_model.pkl`), feature standardization parameters (`liver_scaler.pkl`), packages them into a zip bundle, and displays direct download links.

In [ ]:
# ====================================================================
# 📥 SAVE & DOWNLOAD TRAINED ARTIFACTS ON KAGGLE
# ====================================================================
import os
import pickle
import zipfile
from IPython.display import FileLink, display

saved_files = []

# 1. Save Trained Models
# Save Random Forest
if 'rand_clf' in globals():
    rf_path = 'liver_rf_model.pkl'
    with open(rf_path, 'wb') as f:
        pickle.dump(rand_clf, f)
    saved_files.append(rf_path)
    print(f"✓ Saved Random Forest model: {rf_path}")

# Save Logistic Regression
if 'lr' in globals():
    lr_path = 'liver_lr_model.pkl'
    with open(lr_path, 'wb') as f:
        pickle.dump(lr, f)
    saved_files.append(lr_path)
    print(f"✓ Saved Logistic Regression model: {lr_path}")

# Save Gradient Boosting
if 'gbc' in globals():
    gbc_path = 'liver_gbc_model.pkl'
    with open(gbc_path, 'wb') as f:
        pickle.dump(gbc, f)
    saved_files.append(gbc_path)
    print(f"✓ Saved Gradient Boosting model: {gbc_path}")

# Save Primary Model as liver_model.pkl
primary_model = globals().get('rand_clf', globals().get('lr', None))
if primary_model is not None:
    primary_path = 'liver_model.pkl'
    with open(primary_path, 'wb') as f:
        pickle.dump(primary_model, f)
    saved_files.append(primary_path)
    print(f"✓ Saved Primary model: {primary_path}")

# 2. Save Normalization Scaler Parameters (train_mean, train_std, feature columns)
scaler_data = {}
if 'train_mean' in globals() and 'train_std' in globals():
    scaler_data['mean'] = train_mean
    scaler_data['std'] = train_std
if 'X' in globals():
    scaler_data['feature_names'] = X.columns.tolist()

if scaler_data:
    scaler_path = 'liver_scaler.pkl'
    with open(scaler_path, 'wb') as f:
        pickle.dump(scaler_data, f)
    saved_files.append(scaler_path)
    print(f"✓ Saved Scaler parameters: {scaler_path}")

# 3. Package all files into a single ZIP archive for 1-click download
zip_filename = 'liver_disease_artifacts.zip'
with zipfile.ZipFile(zip_filename, 'w') as zipf:
    for file_name in saved_files:
        if os.path.exists(file_name):
            zipf.write(file_name, os.path.basename(file_name))
print(f"✓ Packaged bundle: {zip_filename}")

# 4. Generate Kaggle Direct Download Links
print("\n" + "="*50)
print("👉 Click the link below to download your complete model bundle:")
print("="*50)
display(FileLink(zip_filename))

print("\nIndividual file downloads:")
for file_name in saved_files:
    if os.path.exists(file_name):
        display(FileLink(file_name))
